Modify the script from the previous exercise - to generate multiple PDFs based on the data from the `equipment.csv` file. Have the new files named `protocol-1.pdf`, `protocol-2.pdf` etc.

In [ ]:
from datetime import datetime  # for current date
import csv  # read data from the CSV file
from reportlab.pdfgen import canvas  # create PDF files
from reportlab.lib.pagesizes import A4  # standard page size

# General company information used in every protocol
CITY = 'Warszawa'
COMPANY_NAME = 'Sample company name'
COMPANY_ADDRESS = 'Company address'
COMPANY_NIP = '9898767654'
COMPANY_REGON = '565434321'

# Get current date automatically
current_date = datetime.now().strftime('%d.%m.%Y')

# Read all rows from the equipment.csv file
# Each row represents one employee and their assigned equipment
with open('equipment.csv', newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))

# Create a separate PDF document for each employee record
for index, row in enumerate(rows, start=1):
    # Compose the employee name and personal ID from the CSV columns
    employee_name = f"{row['Name']} {row['Surname']}"
    employee_pesel = row['ID']

    # Collect all non-empty equipment items from the columns Item 1..Item 4
    # This avoids blank values in the last columns
    equipment_items = [
        value for value in [
            row.get('Item 1', ''),
            row.get('Item 2', ''),
            row.get('Item 3', ''),
            row.get('Item 4', ''),
        ] if value
    ]

    # Name the output file in order: protocol-1.pdf, protocol-2.pdf, ...
    filename = f'protocol-{index}.pdf'
    width, height = A4
    margin_left = 50
    margin_right = 50

    # Create a PDF canvas for the current employee
    c = canvas.Canvas(filename, pagesize=A4)
    y = height - 60

    # Header: place and date
    c.setFont('Helvetica-Bold', 11)
    c.drawString(margin_right, y, f'{CITY}, {current_date}')
    y -= 40

    # Title of the document
    c.setFont('Helvetica-Bold', 18)
    c.drawCentredString(width / 2, y, 'Employee Equipment Agreement')
    y -= 50

    # Employer section
    c.setFont('Helvetica-Bold', 13)
    c.drawString(margin_left, y, 'Employer')
    y -= 20

    c.setFont('Helvetica', 11)
    c.drawString(margin_left, y, COMPANY_NAME)
    y -= 16
    c.drawString(margin_left, y, COMPANY_ADDRESS)
    y -= 16
    c.drawString(margin_left, y, f'NIP: {COMPANY_NIP}')
    y -= 16
    c.drawString(margin_left, y, f'REGON: {COMPANY_REGON}')
    y -= 35

    # Employee section
    c.setFont('Helvetica-Bold', 13)
    c.drawString(margin_left, y, 'Employee')
    y -= 16
    c.setFont('Helvetica', 11)
    c.drawString(margin_left, y, employee_name)
    y -= 16
    c.drawString(margin_left, y, employee_pesel)
    y -= 35

    # Equipment transfer message for this person
    c.setFont('Helvetica', 11)
    c.drawString(margin_left, y, f'The following equipment was handed over on {current_date}:')
    y -= 20

    # Add each equipment item as a separate bullet line
    for item in equipment_items:
        c.drawString(margin_left, y, f'- {item}')
        y -= 16

    # Space before signatures
    y -= 60

    # Employer signature line
    c.drawString(margin_left, y, '..................')
    y -= 16
    c.setFont('Helvetica', 9)
    c.drawString(margin_left, y, 'Date and employer signature')
    y -= 80

    # Employee signature line
    c.setFont('Helvetica', 11)
    c.drawString(margin_left, y, '..................')
    y -= 16
    c.setFont('Helvetica', 9)
    c.drawString(margin_left, y, 'Date and employee signature')

    # Save the PDF for one employee and print a confirmation
    c.save()
    print(f'Generated {filename}')

# Final message after all files are generated
print(f'All {len(rows)} PDF files were created successfully.')